# 第1日 ノートブック03：コンピュータビジョン AI の種類

# Day 1 Notebook 03: Types of Computer Vision AI

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 画像分類・物体検出・セグメンテーション・異常検知の違いを説明できる | 1. Distinguish classification, detection, segmentation and anomaly detection |
| 2. それぞれの出力が何かを言える | 2. State what each task outputs |
| 3. 製造検査での使い分けを提案できる | 3. Propose which task suits a given inspection need |
| 4. それぞれの長所と限界を説明できる | 4. Explain the strengths and limitations of each |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 65 分 / 65 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第1日ノートブック01・02 を終えていること。 / You have finished Day 1 Notebooks 01 and 02. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第1日 / Day 1](../../COURSE_SYLLABUS.md#day1) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 4 つのタスクの違いを説明した表 | A table explaining the four tasks |
| PVC 検査への提案（演習 3） | Your proposal for PVC inspection (exercise 3) |

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第1日 / Day 1 |
| この日の順番 / Position today | 3 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 65 分 / 65 minutes |
| 前提 / Prerequisites | `Day01_02_Pixels_and_Images.ipynb` |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第1日 / Day 1](../../COURSE_SYLLABUS.md#day1) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


> ### ⚠️ このノートブックの図について / About the figures in this notebook
>
> このノートブックに登場する「検出結果」「ヒートマップ」は **概念図** です。
> AI が実際に予測したものではありません。画像の上に **人が描いた** 説明用の図です。
> 本物の AI による推論は第 3 日以降に行います。
>
> The 'detection results' and 'heatmaps' in this notebook are **conceptual
> illustrations**, drawn by hand on top of images. They are *not* real AI predictions.
> Real model inference starts on Day 3.

---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

#### 図の日本語フォントを設定する
Matplotlib の既定フォントは日本語を表示できません。そのままだと図のタイトルが `□□□` になります。次のセルで、日本語を表示できるフォントを探して設定します。Mac や Linux では自動で別のフォントが選ばれます。
#### Configure Japanese text in figures
Matplotlib's default font cannot draw Japanese characters, so figure titles would appear as `□□□`. The next cell looks for a Japanese font and selects it. On macOS or Linux a different font will be chosen automatically.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def show_png(relative_parts, caption=""):
    """素材フォルダの PNG を表示する / display a PNG from the assets folder."""
    path = ASSETS.joinpath(*relative_parts)
    if not path.is_file():
        print("画像がありません / file not found:", path)
        print("assets\\scripts\\generate_training_assets.py を実行してください。")
        return
    image = Image.open(path)
    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.axis("off")
    if caption:
        plt.title(caption)
    plt.show()
    print("size (width, height) =", image.size, " mode =", image.mode)

---

## Part 1　画像分類 / Image classification

### 同じ 1 枚に対する 4 つの出力 / four outputs from one image

![](../../assets/diagrams/teach_cv_tasks_outputs.png)

図：左から分類・検出・セグメンテーション・異常検知の出力の形（**説明用の模式図**。実際のモデルの推論結果ではありません）。

Figure: the shape of the output for classification, detection, segmentation and anomaly detection, left to right (an **illustrative schematic**, not the result of real model inference).

**気づいてほしいこと**

4 つは **答える質問が違います**。出力の形も次のように決まっています。分類は「何が写っているか」で **1 枚に 1 つのラベル**。検出は「どこに何がいくつ」で **四角＋クラス名＋確からしさ**。セグメンテーションは「どの画素が対象か」で **クラスごとの画素マスク**（インスタンスなら個体ごと）。異常検知は「正常とどれだけ違うか」で **スコア＋場所を示すヒートマップ**。同じ画像でも、返ってくる形が違います。この図は形の違いを見るための模式図で、モデルの出力そのものではありません。

**What to notice**

The four answer **different questions**, and each output has its own shape: classification says what is present and returns **one label for the image**; detection says where and how many and returns **boxes + class names + confidence**; segmentation says which pixels and returns **a per-class pixel mask** (per instance for instance segmentation); anomaly detection says how far from normal and returns **a score + a heatmap of where**. The returned shape differs even for the same image. This figure shows the shapes only; it is a schematic, not model output.

**考えてみてください / Try to answer**

- PVC の黒点検査で「位置と個数も記録したい」なら、どの出力が必要ですか。複数選んでもかまいません。
- If PVC spot inspection must also record position and count, which output do you need? More than one is allowed.


#### 分類とは「画像 1 枚に 1 つの答え」
この例の画像分類は、**画像 1 枚に対して 1 つのラベル** を返すタスクです
（1 枚に 1 つのクラスを割り当てる、シングルラベル分類）。

```text
入力：シートの画像 1 枚
出力：OK または NG（＋ その確からしさ）
```

**長所：** 仕組みが単純で、実装しやすい。1 枚に 1 つのラベルを付ければよいので、
画素ごとの塗り分けより正解データを作りやすい。

**短所：** **どこが** 悪いのか分かりません。「NG」と言うだけで、
「右側に 3 mm の黒点がある」とは教えてくれません。
必要なデータの量は、見え方のばらつき、クラス数、求める精度で変わります。
「分類がいつも一番データが少なくて済む」とは言えません。

#### Classification gives one answer per image
The classification in this example returns **one label per image** — single-label classification,
one class per image.

```text
input : one image of the sheet
output: OK or NG  (plus a confidence value)
```

**Strength:** the mechanism is simple and easy to implement, and one label per image is easier to
annotate than painting every pixel.

**Weakness:** it does not say **where** the problem is. It says "NG" but not "there is a 3 mm dark
spot on the right". How much data you need depends on the variation in appearance, the number of
classes and the accuracy required; classification is not always the least data-hungry task.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show_png(["diagrams", "diagram_classification.png"], "classification: 1 image -> 1 label (conceptual)")

---

## Part 2　物体検出 / Object detection

#### 検出とは「どこに何がいくつ」
物体検出は、**欠陥の位置を四角い枠（バウンディングボックス）で囲み**、
種類と確からしさを返します。

```text
入力：シートの画像 1 枚
出力：[x1, y1, x2, y2] の四角 + 種類（黒点／傷／スジ）+ 確からしさ
```

**長所：** 位置と個数が分かります。不良の記録や、ライン停止の判断に使えます。

**短所：** **設定したクラス名しか出力できません**。学習していない種類の欠陥は、
既知のクラスのどれかに押し込まれるか、確からしさが低いまま見落とされます。
また、四角で囲むので、斜めの傷では余分な領域を含みます。

#### Detection answers where, what and how many
Object detection **surrounds each defect with a rectangle (bounding box)** and returns the class and
a confidence.

```text
input : one image
output: [x1, y1, x2, y2] + class (spot / scratch / streak) + confidence
```

**Strength:** you learn position and count — useful for records and line-stop decisions.

**Weakness:** it can only output **the class names it was configured with**. A defect type it was not
trained on is pushed into one of the known classes, or left with a low score and missed. Also a
rectangle is a crude fit for a diagonal scratch.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show_png(["diagrams", "diagram_detection.png"], "detection: bounding boxes + class + confidence (conceptual)")

---

## Part 3　セグメンテーション / Segmentation

#### セマンティックセグメンテーション（画素ごとの分類）
セグメンテーションは、**画素ごと** に「これは欠陥か、そうでないか」を返します。
マスク画像（白＝欠陥、黒＝正常）が出力です。

**長所：** 欠陥の **面積** が分かります。「面積が 0.5 mm² 以上の欠陥は NG」のように、
境界を含めて判定できます。細長い傷の形もそのまま表現できます。

**短所：** 正解データ（マスク）を作るのに **時間がかかります**。
1 枚の画像に対して、画素単位で塗る作業が必要です。

面積を mm² で扱うには **校正** が必要です。1 画素が何 mm かを先に決めておかないと、
画素数の面積を mm² の規格と比べることになり、判定が狂います（第 2 日以降で扱います）。

#### Semantic segmentation (classifying each pixel)
Segmentation decides, **pixel by pixel**, whether it is defect or not. The output is a mask
(white = defect, black = normal).

**Strength:** you get the defect **area**, enabling an inclusive rule such as "reject if the area is
0.5 mm² or more". Thin scratches keep their shape.

**Weakness:** ground-truth masks are **expensive to create** — someone must paint every pixel.

Reporting the area in mm² needs **calibration**: decide how many millimetres one pixel covers
(covered from Day 2). Comparing a raw pixel count against an mm² limit gives a wrong decision.


### 意味とインスタンスの違い / semantic versus instance

![](../../assets/diagrams/teach_segmentation_modes.png)

図：同じ 2 つの欠陥を、意味セグメンテーション（1 つのクラスにまとめる）とインスタンスセグメンテーション（1 つずつ番号を付ける）で表したもの（合成画像）。

Figure: the same two defects under semantic segmentation (one class) and instance segmentation (numbered one by one) — synthetic image.

**気づいてほしいこと**

セマンティックセグメンテーションは画素ごとのクラスを返すので、同じクラスの領域が離れていても数えられます。区別が必要なのは**接している個体**です。その場合はインスタンスセグメンテーションを使います。

**What to notice**

Semantic segmentation returns a class per pixel, so separated regions of the same class can still be counted. What needs a different method is **touching instances**, which is what instance segmentation separates.

**考えてみてください / Try to answer**

- 接していない 2 つの黒い汚れは、セマンティックセグメンテーションで 2 つとして数えられますか。
- Can two separated dark marks be counted as two with semantic segmentation?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show_png(["diagrams", "diagram_segmentation.png"], "semantic segmentation: a mask (conceptual)")

#### インスタンスセグメンテーション
セマンティックセグメンテーションが「全部まとめて欠陥」とするのに対し、
インスタンスセグメンテーションは **欠陥 1 つ 1 つを区別** します。

```text
セマンティック：  欠陥 3 個 → 1 つのマスク（3 つの領域）
インスタンス：    欠陥 3 個 → マスク 1・マスク 2・マスク 3（それぞれ別の色）
```

セマンティックのマスクでも、**離れている領域** は連結成分として数えられます。
数えられないのは **接している欠陥** です。2 つの欠陥が 1 画素でも触れていると、
セマンティックのマスクでは 1 つの領域に融合し、区別できません。
ここがインスタンスセグメンテーションとの本当の違いです。

離れた欠陥の個数を数えるだけなら、セグメンテーションと連結成分の数え上げで足ります。
接している欠陥も 1 つずつ区別したいときに、インスタンスセグメンテーションが必要になります。

#### Instance segmentation
Whereas semantic segmentation merges everything into "defect", instance segmentation **separates
each individual defect**.

```text
semantic:  3 defects -> one mask (three separate regions)
instance:  3 defects -> mask 1, mask 2, mask 3 (each a different colour)
```

A semantic mask **can** still count **separated** regions by connected-component labelling. What it
cannot separate is **touching** defects: if two defects touch even at one pixel they merge into one
region in the mask. That is the real difference from instance segmentation.

To count separated defects, semantic segmentation plus connected components is enough. To tell
touching defects apart, instance segmentation is needed.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show_png(["diagrams", "diagram_instance_segmentation.png"], "instance segmentation: one colour per defect (conceptual)")

---

## Part 4　異常検知 / Anomaly detection

#### 異常検知は「いつもと違う」を探す
この研修で使う PatchCore のような異常検知は、**正常品の画像だけ** から
「正常とはどういうものか」を学び、それと違う部分を **異常スコア** として数値で返します。

```text
入力：シートの画像 1 枚
出力：異常スコア（数値）+ 異常の場所を示すヒートマップ
```

**長所：** 学習に **欠陥の種類を教えなくてもよい** のが最大の長所です。
「黒点」「傷」などとラベルを付けた異常画像を集めなくても、正常品だけで学習できます。
ラベル付きの異常画像は、学習ではなく **しきい値の較正と評価** に使います。

**短所：** **何の欠陥かは分かりません**。また、「正常のばらつき」を異常と判定してしまう
ことがあります（色ムラ、模様、照明のムラなど）。

ヒートマップは **場所の手がかり** を返しますが、欠陥の種類や正確な輪郭は分かりません。

> 第 5 日で PatchCore という異常検知の手法を実際に動かします。

#### Anomaly detection looks for 'not like usual'
Anomaly detection of the kind used here — PatchCore — is fitted on **normal images only** and reports
how far a new image deviates, as an **anomaly score**.

```text
input : one image
output: anomaly score (a number) + a heatmap showing where
```

**Strength:** you do **not** need defect labels for fitting. You do not have to collect labelled
defect images to train; normal images are enough. Labelled defects are then used for **threshold
calibration and evaluation**, not for fitting.

**Weakness:** it cannot tell you **which** defect it is, and it may call acceptable surface variation
(colour mottling, pattern, uneven lighting) an anomaly.

The heatmap gives a hint about **where**, but not the defect class or an exact outline.

Day 5 runs this method, called PatchCore.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show_png(["diagrams", "diagram_anomaly_heatmap.png"], "anomaly score map + heatmap (conceptual)")

#### 4 つのタスクの比較
ここまでの 4 つを 1 つの表にまとめます。

| タスク | 出力 | 必要な正解データ | 主な長所 | 主な短所 |
|---|---|---|---|---|
| 画像分類 | 1 枚に 1 ラベル | 画像ごとの OK/NG | 仕組みが単純 | 位置が分からない |
| 物体検出 | 四角＋クラス名＋確からしさ | 四角の座標＋クラス | 位置と個数が分かる | 設定したクラス名しか出せない |
| セマンティックセグメンテーション | クラスごとの画素マスク | 画素ごとの塗り分け | 面積・形が分かる | 正解データ作成が大変。接した欠陥は区別できない |
| 異常検知 | スコア＋ヒートマップ | 正常品の画像（＋較正用の異常画像） | 学習に欠陥ラベルが要らない | 種類は分からない。誤検知しやすい |

#### Comparing the four tasks
All four in one table.

| Task | Output | Ground truth needed | Main strength | Main weakness |
|---|---|---|---|---|
| Classification | one label per image | OK/NG per image | simple mechanism | no location |
| Detection | box + class name + confidence | box coordinates + class | location and count | can only name configured classes |
| Semantic segmentation | per-class pixel mask | pixel-level painting | area and shape | expensive to annotate; touching defects merge |
| Anomaly detection | score + heatmap | normal images (+ abnormal images for calibration) | no defect labels needed for fitting | no class; more false alarms |


---

## Part 5　製造での使い分け / Choosing in manufacturing

### 実在の産業向け異常検知データの例 / a real industrial anomaly-detection dataset

![](../../assets/external_samples/open_reference/VisA_samples.png)

図：VisA データセットの概要（Yang Zou ほか / Amazon Science、VisA データセットは CC BY 4.0）。回路基板・部品・カプセルなど、**工業製品の実写**が並びます。この研修の PVC 製品ではなく、この研修のモデルの出力でもありません。

Figure: the VisA overview (Yang Zou et al. / Amazon Science; VisA dataset, CC BY 4.0). Real photographs of industrial objects — circuit boards, parts, capsules. These are not this course's PVC products and not outputs of its models.

**気づいてほしいこと**

欠陥の見え方は製品ごとに違います。明るさの平均 1 つでは表せない差（形・模様・位置）が、どの製品にもあります。

**What to notice**

What counts as an anomaly differs by product. In every product there are differences — shape, pattern, position — that one average brightness value cannot describe.

**考えてみてください / Try to answer**

- この中で、画像全体の平均の明るさを比べるだけでは見つけにくい欠陥はどれだと思いますか。
- Which of these defects do you think a whole-image average brightness comparison would miss most easily?


#### 現場の質問から選ぶ
技術から選ぶのではなく、**現場の質問** から選びます。

| 現場の質問 | 向いているタスク |
|---|---|
| 「この製品は出荷してよいか」 | 画像分類、または異常検知 |
| 「どこに、何が、いくつあるか知りたい」 | 物体検出 |
| 「不良の面積を測りたい」 | セグメンテーション |
| 「まだ見たことのない不良も捕まえたい」 | 異常検知（ただし万能ではありません） |
| 「不良の発生傾向をデータで見たい」 | 物体検出（種類と個数が記録できる） |

実際の現場では、**複数の手法を組み合わせる** ことが多いです。
たとえば「異常検知で怪しい製品を絞り、物体検出で種類を判定する」などです。

**どの出力も、単独では「最良」ではありません。** 検査の要求と出力の形が合うかどうかで選びます。
要求が 2 つ以上あるときは、1 つの出力で済ませようとせず、組み合わせます。

| 検査の要求 / requirement | 合う出力 / matching output | その出力でできないこと / what it cannot do |
|---|---|---|
| 出荷可否を速く知りたい / a fast ship-or-not answer | 画像分類（1 枚に 1 ラベル）、または異常検知（スコア） | 分類はどこが悪いかを言えない。異常検知は種類を言えない |
| 位置と個数を記録したい / position and count | 物体検出（四角＋クラス名） | 面積は測れない（四角で近似） |
| 不良の面積・形を測りたい / area and shape | セグメンテーション | 正解付けの手間が大きい。接した欠陥は区別できない |
| 見たことのない不良も拾いたい / catch unseen defects | 異常検知 | 種類は言えない。誤報が増えやすい |

#### Start from the shop-floor question
Choose from the **question**, not from the technology.

| Shop-floor question | Suitable task |
|---|---|
| "may this product ship?" | classification or anomaly detection |
| "where, what, how many?" | object detection |
| "how large is the defect area?" | segmentation |
| "catch defects we have never seen" | anomaly detection (not a guarantee) |
| "analyse defect trends over time" | object detection (gives class and count) |

In practice plants **combine** methods, e.g. anomaly detection to shortlist suspicious products,
then object detection to name the defect type.

**No single output is best on its own.** Choose by whether the shape of the output meets the
inspection requirement; when there are two or more requirements, combine methods.

| requirement | matching output | what it cannot do |
|---|---|---|
| a fast ship-or-not answer | classification (one label per image) or anomaly detection (a score) | classification gives no location; anomaly detection gives no class |
| position and count | object detection (box + class name) | no area (a box is an approximation) |
| area and shape | segmentation | expensive annotation; touching defects merge |
| catch unseen defects | anomaly detection | no class; more false alarms |


---

## Part 6　生成したサンプル画像で確かめる / Checking with a generated example image


#### 合成シートの欠陥を見る
生成済みの合成画像を見て、4 つのタスクならそれぞれ何を返すか考えてください。
この画像は **私たちが Python で描いた** もので、実際の工場画像ではありません。

#### Look at the synthetic defects
Look at the generated synthetic image and think about what each of the four tasks
would return. This image was **drawn by our own Python code**; it is not a factory image.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json
from pathlib import Path

defect_image = ASSETS / "defect_samples" / "defect_multi.png"
mask_image = ASSETS / "defect_samples" / "defect_multi_mask.png"
records = json.loads((ASSETS / "defect_samples" / "defects.json").read_text(encoding="utf-8"))

if not defect_image.is_file():
    print("素材がありません / missing asset:", defect_image)
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(Image.open(defect_image)); axes[0].set_title("image / 入力画像"); axes[0].axis("off")
    axes[1].imshow(Image.open(mask_image), cmap="gray"); axes[1].set_title("mask / 正解マスク"); axes[1].set_xticks([]); axes[1].set_yticks([])

    overlaid = np.asarray(Image.open(defect_image)).copy()
    for entry in records:
        if entry["image"] != "defect_multi.png":
            continue
        for defect in entry["defects"]:
            x1, y1, x2, y2 = defect["bbox"]
            overlaid[y1:y2, x1:x1 + 2] = [255, 0, 0]
            overlaid[y1:y2, x2 - 2:x2] = [255, 0, 0]
            overlaid[y1:y1 + 2, x1:x2] = [255, 0, 0]
            overlaid[y2 - 2:y2, x1:x2] = [255, 0, 0]
    axes[2].imshow(overlaid); axes[2].set_title("hand-drawn boxes / 人が描いた枠"); axes[2].axis("off")
    plt.tight_layout(); plt.show()

    print("この画像に含まれる欠陥（正解データ）/ defects in this image (ground truth):")
    for entry in records:
        if entry["image"] != "defect_multi.png":
            continue
        for defect in entry["defects"]:
            print(" ", defect["cls"], defect["bbox"])

### 実行結果の例 / Expected output

この結果は `assets/defect_samples/defects.json` に保存された正解データから作られます
（同じ素材なら同じ結果です）。見出しはコードが出力する日本語のままです。

```text
この画像に含まれる欠陥（正解データ）/ defects in this image (ground truth):
  scratch [123, 23, 147, 65]
  streak [174, 118, 308, 125]
  black_spot [138, 138, 151, 149]
```

These values come from the ground truth stored in `assets/defect_samples/defects.json` (the same
assets give the same result). The header is the Japanese text printed by the code.

The three classes are `scratch`, `streak` and `black_spot`; the four numbers are the box as
[x1, y1, x2, y2] in pixels.


#### 4 つのタスクの出力（セグメンテーションは 2 種類）/ what each task returns

上の画像に対して、4 つのタスクは次のものを返します。セグメンテーションだけは、意味（クラスごと）とインスタンス（個体ごと）の 2 種類を並べています。

| タスク | 返ってくるもの |
|---|---|
| 画像分類 | 「NG」（1 枚に 1 つのラベル） |
| 物体検出 | 3 つの四角と、それぞれのクラス名（scratch / streak / black_spot） |
| セマンティックセグメンテーション | 欠陥クラスのマスク（離れた 3 つの領域）と、それぞれの面積 |
| インスタンスセグメンテーション | 欠陥ごとに分かれた 3 つのマスクと、それぞれの面積 |
| 異常検知 | 1 つのスコア（例 0.87）と、場所を示すヒートマップ |

#### What would each task return here

For the image above the tasks return the following. Segmentation has two rows because it comes in two kinds: semantic (per class) and instance (per object).

| Task | What comes back |
|---|---|
| Classification | "NG" (one label for the image) |
| Detection | three boxes, each with its class name (scratch / streak / black_spot) |
| Semantic segmentation | a defect-class mask with three separated regions, and their areas |
| Instance segmentation | three separate masks, one per defect, and their areas |
| Anomaly detection | one score (e.g. 0.87) and a heatmap of where |


---

## 演習 / Exercises

#### 演習 1（基本）：4 つのタスクを自分の言葉で説明する
下のセルに、4 つのタスクを 1 行ずつ説明してください。
「出力は何か」「どんなときに使うか」を含めてください。

**完了の目安:** 4 行それぞれに、出力の形と、製造での具体的な使い道が書かれている。

#### Exercise 1 (basic): explain the four tasks in your own words
In the cell below, describe each of the four tasks in one line, including
"what is the output" and "when would I use it".

**Done when:** each line names the output and a concrete manufacturing use.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
classification = ""
detection = ""
segmentation = ""
anomaly_detection = ""

for name, text in [("classification", classification), ("detection", detection),
                   ("segmentation", segmentation), ("anomaly", anomaly_detection)]:
    print(f"{name:<16} {text or '(未記入 / not written)'}")

#### 演習 2（中級）：現場の質問に答える
次の 3 つの質問に対して、どのタスクを選ぶかと、その理由を書いてください。

1. 「このロールは出荷できるか。とにかく見逃したくない。」
2. 「今月、黒点が何個あったか数えたい。」
3. 「傷の面積が 1 mm² を超えたら NG にしたい。」

**完了の目安:** 各質問について、(1) 選んだタスク、(2) その出力が要求を満たす理由、
(3) その選択に残るトレードオフ（できなくなること、増える手間や誤報）を書く。

#### Exercise 2 (intermediate): answer shop-floor questions
For each question, choose a task and justify it.

1. "Can this roll ship? Above all I must not miss a defect."
2. "I want to count how many dark spots occurred this month."
3. "Reject when the scratch area exceeds 1 mm²."

**Done when:** for each question you name the task, explain why its output meets the requirement,
and state the trade-off that remains (what it cannot do, or what extra cost it brings).


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
answers = {
    "1. 出荷可否 / ship or not": ("", ""),
    "2. 黒点の件数 / count spots": ("", ""),
    "3. 面積で判定 / area rule": ("", ""),
}
for question, (task, reason) in answers.items():
    print(f"{question}\n   task: {task or '-'}\n   why : {reason or '-'}\n")

#### 演習 3（発展・任意）：PVC 検査への提案
幅 1,300 mm、欠陥 0.3 mm、高速連続生産という条件で、
**どのタスクを最初に試すべきか** を提案してください。次の観点を含めてください。

- 最初に試すタスクと、その理由
- 必要なデータ（何を、何枚くらい）
- うまくいかない可能性が高い点
- 第 2 週で確認すべきこと（カメラ・照明・速度）

**完了の目安:** 最初のタスク、データ計画、リスク、第 2 週に確かめることを、それぞれ書く。

#### Exercise 3 (optional challenge): propose an approach for PVC inspection
Given a 1,300 mm wide sheet, 0.3 mm defects and high-speed continuous production,
propose **which task to try first**. Address:

- the task you would try first and why
- the data you would need (what, roughly how many)
- what is most likely to fail
- what must be checked in Week 2 (camera, lighting, speed)

**Done when:** your proposal states a first task, a data plan, a risk and a Week 2 question.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
first_task = ""
reason = ""
data_plan = ""
risks = ""
week2_checks = ""

print("最初のタスク / first task:", first_task or "(未記入 / not written)")
print("理由 / reason       :", reason or "(未記入 / not written)")
print("データ計画 / data   :", data_plan or "(未記入 / not written)")
print("リスク / risks      :", risks or "(未記入 / not written)")
print("第2週の確認 / week2 :", week2_checks or "(未記入 / not written)")

---

#### 理解度チェック / Understanding questions

1. 画像分類と物体検出の、出力の違いを説明してください。
2. セマンティックとインスタンスのセグメンテーションは、何が違いますか。
3. この研修の異常検知（PatchCore）は、なぜ学習に欠陥の種類を教えなくてよいのですか。
   その代わり、何が分かりませんか。
4. 現場で「面積で判定したい」と言われたら、どのタスクを選びますか。何を用意する必要がありますか。
5. 4 つのタスクのうち、正解データ（ラベル）を作るのが最も大変なのはどれですか。理由も書いてください。

1. Explain the difference in output between classification and detection.
2. What separates semantic from instance segmentation?
3. Why can the anomaly detection used in this course (PatchCore) be fitted without defect labels,
   and what does it therefore not tell you?
4. Which task would you choose if asked to "judge by area", and what must you prepare for it?
5. Which of the four tasks is the most expensive to annotate, and why?

Model answers: `docs/instructor/day01_instructor_guide.md`.


---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| 素材画像が表示されない | The asset image does not appear | `python assets\scripts\generate_training_assets.py` を実行して素材を生成してください。 | Run `python assets\scripts\generate_training_assets.py` to create the assets. |
| `json.decoder.JSONDecodeError` | `json.decoder.JSONDecodeError` | `defects.json` が壊れています。素材生成スクリプトを `--clean` 付きで再実行してください。 | `defects.json` is corrupted. Re-run the generator with `--clean`. |
| 図の内容が、AI の結果と違って見える | A figure does not look like AI output | 正しい認識です。このノートブックの図は概念図で、AI の予測ではありません。 | Correct: the figures are conceptual illustrations, not predictions. |
| `KeyError: 'bbox'` | `KeyError: 'bbox'` | `defects.json` の形式が古い可能性があります。素材を作り直してください。 | The JSON format may be outdated. Regenerate the assets. |

## 完了チェックリスト / Completion checklist

**必須** の項目を並べています。すべてにチェックが付いたら、このノートブックは完了です。

Only the **required** items are listed. This notebook is complete when every box is checked.

- [ ] Part 1〜Part 6 を読んだ / Read Parts 1-6
- [ ] すべてのコードセルを実行した / Ran all code cells
- [ ] 概念図が AI の予測ではないことを理解した / Understood the figures are not AI predictions
- [ ] 演習 1〜2（必須）を実施した / Completed exercises 1-2 (required)
- [ ] 4 つのタスクの違いを、人に説明できる / Can explain the four tasks to a colleague
- [ ] 理解度チェックの 5 問に答えた / Answered the 5 questions
- [ ] ノートブックを保存した / Saved the notebook

**任意（完了条件には含みません）:** 演習 3、さらに学ぶの追加チャレンジ。

**Optional, not part of completion:** Exercise 3 and the extra challenge under Further learning.


## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics: 物体検出のタスク](https://docs.ultralytics.com/tasks/detect/)
- [Ultralytics: セグメンテーション](https://docs.ultralytics.com/tasks/segment/)
- [Anomalib ドキュメント](https://anomalib.readthedocs.io/)
- [OpenCV 画像処理チュートリアル](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

「異常検知」と「物体検出」を両方使うとしたら、どんな順番で使うのがよいか、不良品を 1 つも出さないという目的で考えてみてください。

If you combined anomaly detection and object detection, in what order would you apply them to avoid shipping any defective product?